# 08 · Observability: structured logging and tracing

Log every LLM call as one structured JSON line with the fields you will need at 3 a.m., build a tiny tracer that
turns an agent run into a waterfall of nested spans, compute p95 / error rate / tokens per day from logs with
pandas, write the alert rules worth having, and detect drift.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [05 · Latency percentiles and cost](05_latency_percentiles_p50_p95_and_cost.ipynb), [pandas essentials](../04_numpy_pandas_pytorch/02_pandas_essentials.ipynb)

## Why this matters in interviews

- "What does an LLM observability stack look like?" and "which alerts would you configure?" come up in every
  production round. A wrong answer in an LLM system is still HTTP 200, so request-level logs are not enough: you
  need traces, cost, and quality signals.
- "Token usage doubled overnight with no deploy" and "the product feels slow" are debugging questions whose answer
  starts with "here is what I logged, and here is the query".
- The privacy trap (full prompts with PII shipped to a tracing vendor) is the most common failure in this stack and
  interviewers check that you name it.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pd06` | What does an LLM observability stack look like? |
| `tl05` | How would you add observability to an LLM application? |
| `ev13` | What is LLM observability, and what would you instrument? |
| `ag26` | What does an agent observability stack look like? |
| `po23` | What would you put on a monitoring dashboard for a GenAI system? |
| `po24` | Which alerts would you actually configure, and which would you not? |
| `pd07` | What alerts would you set up for an LLM feature? |
| `po32` | Token usage doubled overnight with no deploy. How do you find the cause? |
| `pd30` | What is drift in an LLM system, and how do you detect it? |
| `tl32` | Compare hosted and self-hosted observability for an LLM app. |

## Setup

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import contextvars
import functools
import hashlib
import io
import json
import logging
import re
import time
import uuid
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import openai
import pandas as pd
from llm_setup import OFFLINE, make_client

client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=5)  # warm-up
print(f"provider={PROVIDER}, pandas {pd.__version__}")

## 1. What to record for every LLM call

| Field | Why you will want it |
|---|---|
| `trace_id`, `span_id`, `request_id` | join everything about one user request; the id shown in user-facing errors |
| `model` (requested and returned), `prompt_version`, params | "did the prompt, the model or the provider change?" |
| `route` / tier, `cache_hit`, `fallback` | which path served it; never mix degraded answers into quality stats |
| `input_tokens`, `cached_tokens`, `output_tokens` | cost, prompt-cache health, runaway prompts |
| `latency_ms`, `ttft_ms` | percentiles per model and stage |
| `cost_usd` | cost per request p50/p99, per feature, per tenant |
| `status`, `error_class`, `retries` | error rate by class: 429 and 5xx mean different things |
| retrieved chunk ids + scores, tool calls, guardrail verdicts | "what did we retrieve?" is the first question on every bad answer |
| `tenant`, hashed `user` | per-tenant limits and debugging, without storing identities |

**Never** log raw PII or secrets. Log prompts redacted, or as a hash plus the template version (full prompts, if
at all, in a separate store with access control and short retention). Tracing tools capture full prompts **by
default**, often in another jurisdiction (`pd06`, `po23`).

## 2. Structured JSON logging with the standard `logging` module

One JSON object per line is what log pipelines (CloudWatch, Datadog, Loki, BigQuery) index well. A custom
`Formatter` does it, and a `contextvars` variable carries the request id into every log line without passing it
through every function.

In [ ]:
request_id_var = contextvars.ContextVar("request_id", default="-")

class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        entry = {"ts": datetime.fromtimestamp(record.created, timezone.utc).isoformat(timespec="milliseconds"),
                 "level": record.levelname, "logger": record.name, "msg": record.getMessage(),
                 "request_id": request_id_var.get()}
        entry.update(getattr(record, "fields", {}))                     # structured extras
        if record.exc_info:
            entry["error"] = self.formatException(record.exc_info).splitlines()[-1]
        return json.dumps(entry, default=str)

log_buffer = io.StringIO()                                              # stands in for stdout / a log shipper
handler = logging.StreamHandler(log_buffer)
handler.setFormatter(JsonFormatter())
log = logging.getLogger("chat-service")
log.handlers[:] = [handler]; log.setLevel(logging.INFO); log.propagate = False

request_id_var.set("req-7f3a")
log.info("request received", extra={"fields": {"route": "/chat", "tenant": "acme"}})
try:
    1 / 0
except ZeroDivisionError:
    log.exception("post-processing failed", extra={"fields": {"stage": "format_answer"}})
for line in log_buffer.getvalue().splitlines():
    print(line)
assert all(json.loads(l)["request_id"] == "req-7f3a" for l in log_buffer.getvalue().splitlines())

### An instrumented LLM call

Wrap the client once and every call emits the same record. PII is masked before it reaches the log (a minimal
email masker here; use the redactor from [notebook 07](07_guardrails_pii_and_prompt_injection.ipynb) in real code).

In [ ]:
PRICES = {"gpt-4.1-mini": (0.40, 0.10, 1.60)}                        # $ per 1M: input, cached input, output
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")

def redact(text: str) -> str:
    return EMAIL_RE.sub("<EMAIL>", text)

def logged_chat(messages, *, llm=client, model=MODEL, prompt_version="support-v7", **params):
    t0 = time.perf_counter()
    fields = {"event": "llm_call", "model": model, "prompt_version": prompt_version, "params": params,
              "prompt_hash": hashlib.sha256(json.dumps(messages).encode()).hexdigest()[:12],
              "last_user_msg": redact(messages[-1]["content"])[:200]}
    try:
        r = llm.chat.completions.create(model=model, messages=messages, **params)
    except openai.APIError as e:
        fields.update(status="error", error_class=type(e).__name__, latency_ms=round(1000 * (time.perf_counter() - t0), 1))
        log.error("llm call failed", extra={"fields": fields})
        raise
    u = r.usage
    cached = (u.prompt_tokens_details.cached_tokens if u.prompt_tokens_details else 0) or 0
    p_in, p_cached, p_out = PRICES.get(model, (0.0, 0.0, 0.0))
    fields.update(status="ok", response_model=r.model, finish_reason=r.choices[0].finish_reason,
                  input_tokens=u.prompt_tokens, cached_tokens=cached, output_tokens=u.completion_tokens,
                  latency_ms=round(1000 * (time.perf_counter() - t0), 1),
                  cost_usd=round(((u.prompt_tokens - cached) * p_in + cached * p_cached + u.completion_tokens * p_out) / 1e6, 8))
    log.info("llm call", extra={"fields": fields})
    return r.choices[0].message.content

log_buffer.seek(0); log_buffer.truncate()
request_id_var.set("req-81c2")
logged_chat([{"role": "user", "content": "I'm jane.doe@example.com. What is 17*23?"}], max_tokens=30)
if OFFLINE:
    OFFLINE.fail_next(1, 429)
try:
    logged_chat([{"role": "user", "content": "What is 2+2?"}], llm=make_client(max_retries=0), max_tokens=10)
except openai.APIError:
    pass
records = [json.loads(l) for l in log_buffer.getvalue().splitlines()]
for rec in records:
    print(json.dumps(rec, indent=None)[:330])
assert "jane.doe@example.com" not in log_buffer.getvalue(), "no raw PII in logs"

## 3. A tiny tracer: nested spans for an agent run

A **trace** is one request; a **span** is one timed step inside it with a parent, so steps nest (agent run >
retrieve, llm call, tool call, llm call). This is the whole idea behind OpenTelemetry, LangSmith and Langfuse. A
`contextvars` variable holds the current span, so children find their parent automatically, even across `await`.

In [ ]:
current_span = contextvars.ContextVar("current_span", default=None)
FINISHED = []                                                          # a real tracer exports these in batches

class Span:
    def __init__(self, name, **attributes):
        parent = current_span.get()
        self.name, self.attributes, self.events = name, dict(attributes), []
        self.trace_id = parent.trace_id if parent else uuid.uuid4().hex[:16]
        self.span_id, self.parent_id = uuid.uuid4().hex[:8], parent.span_id if parent else None
        self.status = "ok"
    def __enter__(self):
        self.start, self._token = time.perf_counter(), current_span.set(self)
        return self
    def __exit__(self, exc_type, exc, tb):
        self.end = time.perf_counter()
        if exc is not None:
            self.status = "error"; self.attributes["error.type"] = type(exc).__name__
        current_span.reset(self._token)
        FINISHED.append(self)
        return False                                                   # never swallow the exception
    def set(self, **attrs):
        self.attributes.update(attrs)

def traced(name=None):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            with Span(name or fn.__name__):
                return fn(*args, **kwargs)
        return wrapper
    return deco

In [ ]:
if OFFLINE:
    OFFLINE.latency = 0.25                                                # offline: model calls take 0.25 s

@traced("retrieve")
def retrieve(query):
    time.sleep(0.12)                                                      # stands in for a vector search
    current_span.get().set(**{"retrieval.k": 3, "retrieval.chunk_ids": ["kb-12", "kb-40", "kb-41"]})
    return "The X200 has four Ethernet ports. Firmware updates ship quarterly. Support is open 9-6."

def llm_step(step, messages):
    with Span("llm.chat", **{"gen_ai.request.model": MODEL, "step": step}) as sp:
        r = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=40)
        sp.set(**{"gen_ai.usage.input_tokens": r.usage.prompt_tokens, "gen_ai.usage.output_tokens": r.usage.completion_tokens})
        return r.choices[0].message.content

@traced("tool.get_order_status")
def get_order_status(order_id):
    time.sleep(0.2)                                                       # stands in for an internal API
    return {"order_id": order_id, "status": "shipped"}

def agent_run(question):
    with Span("agent.run", **{"user.question": redact(question)}) as root:
        context = retrieve(question)
        plan = llm_step("plan", [{"role": "user", "content": f"Context: {context}\nQuestion: {question}"}])
        order = get_order_status("A-100")
        answer = llm_step("answer", [{"role": "user", "content": f"Summarise this.\n\nOrder {order['order_id']} is "
                                      f"{order['status']}. {context}"}])
        root.set(steps=4)
        return answer, root.trace_id

FINISHED.clear()
answer, trace_id = agent_run("Has my order A-100 shipped, and how many ports does the X200 have?")
spans = sorted(FINISHED, key=lambda s: s.start)
t_start = spans[0].start
depth = {s.span_id: 0 for s in spans}
for s in spans:
    depth[s.span_id] = depth.get(s.parent_id, -1) + 1
    print(f"{'  ' * depth[s.span_id]}{s.name:<22} {1000 * (s.start - t_start):6.0f} ms -> {1000 * (s.end - t_start):6.0f} ms  "
          f"span={s.span_id} parent={s.parent_id}")
assert len({s.trace_id for s in spans}) == 1 and sum(s.parent_id is None for s in spans) == 1

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
colors = {"agent": "#adb5bd", "retrieve": "#74c0fc", "llm": "#b197fc", "tool": "#ffa94d"}
for i, s in enumerate(spans):
    c = next(v for k, v in colors.items() if s.name.startswith(k))
    ax.barh(i, 1000 * (s.end - s.start), left=1000 * (s.start - t_start), color=c, edgecolor="#495057")
    ax.text(1000 * (s.end - t_start) + 8, i, f"{1000 * (s.end - s.start):.0f} ms", va="center", fontsize=8)
ax.set_yticks(range(len(spans)), [("   " * depth[s.span_id]) + s.name for s in spans]); ax.invert_yaxis()
ax.set_xlim(0, 1150 * (spans[0].end - t_start))
ax.set(xlabel="ms since the request started", title=f"Trace {trace_id}: where the time went")
plt.show()

The waterfall answers "why is it slow?" in one glance: steps run one after another, and the two LLM calls dominate.
Independent steps (retrieval and the order lookup) could run concurrently ([notebook 01](01_async_concurrency_for_llm_calls.ipynb)).
Per step, log the attributes shown (model, tokens, chunk ids); keep **all** traces for failed runs and a sample of
successful ones.

## 4. Metrics from logs with pandas

A week of synthetic call logs for three models (6,000 calls), with two things hidden in it: an error spike on one
model on day 5, and a prompt that quietly got longer from day 4. We compute what a dashboard shows.

In [ ]:
rng = np.random.default_rng(0)
n = 6_000
start = pd.Timestamp("2026-09-01", tz="UTC")
ts = start + pd.to_timedelta(np.sort(rng.uniform(0, 7 * 86_400, n)), unit="s")
model = rng.choice(["gpt-4.1-mini", "gpt-4.1", "gpt-4.1-nano"], n, p=[0.6, 0.15, 0.25])
base_ms = pd.Series(model).map({"gpt-4.1-mini": 900, "gpt-4.1": 1_800, "gpt-4.1-nano": 500}).to_numpy()
latency_ms = base_ms * rng.lognormal(0, 0.35, n) + np.where(rng.random(n) < 0.02, rng.uniform(3_000, 9_000, n), 0)
day = (ts - start).days
prompt_version = np.where(day >= 3, np.where(rng.random(n) < 0.7, "support-v8", "support-v7"), "support-v7")
input_tokens = np.where(prompt_version == "support-v8", 2_600, 1_200) + rng.integers(0, 400, n)
output_tokens = rng.integers(60, 300, n)
err_p = np.where((day == 4) & (model == "gpt-4.1"), 0.25, 0.01)
status = np.where(rng.random(n) < err_p, rng.choice(["RateLimitError", "InternalServerError", "APITimeoutError"], n,
                                                     p=[0.6, 0.3, 0.1]), "ok")
price = pd.Series(model).map({"gpt-4.1-mini": (0.4, 1.6), "gpt-4.1": (2.0, 8.0), "gpt-4.1-nano": (0.1, 0.4)})
cost = np.array([(i * p[0] + o * p[1]) / 1e6 for i, o, p in zip(input_tokens, output_tokens, price)])
logs = pd.DataFrame({"ts": ts, "model": model, "prompt_version": prompt_version, "latency_ms": latency_ms,
                     "input_tokens": input_tokens, "output_tokens": output_tokens, "status": status, "cost_usd": cost})
logs["day"] = logs["ts"].dt.strftime("%m-%d")
print(logs.head(3).to_string())
print(logs.dtypes.to_dict())

In [ ]:
per_model = logs.groupby("model").agg(
    calls=("latency_ms", "size"),
    p50_ms=("latency_ms", lambda s: s.quantile(0.50)),
    p95_ms=("latency_ms", lambda s: s.quantile(0.95)),
    p99_ms=("latency_ms", lambda s: s.quantile(0.99)),
    error_rate=("status", lambda s: (s != "ok").mean()),
    cost_per_call_p99=("cost_usd", lambda s: s.quantile(0.99)),
).round(4)
print(per_model.to_string())
daily = logs.groupby("day").agg(calls=("status", "size"), error_rate=("status", lambda s: (s != "ok").mean()),
                                input_tokens=("input_tokens", "sum"), output_tokens=("output_tokens", "sum"),
                                cost_usd=("cost_usd", "sum"))
print(daily.round(3).to_string())
assert per_model.loc["gpt-4.1", "p95_ms"] > per_model.loc["gpt-4.1-nano", "p95_ms"]

In [ ]:
p95_daily = logs.pivot_table(index="day", columns="model", values="latency_ms", aggfunc=lambda s: s.quantile(0.95))
err_daily = logs.assign(err=logs["status"] != "ok").pivot_table(index="day", columns="model", values="err", aggfunc="mean")
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
p95_daily.plot(ax=axes[0], marker="o"); axes[0].set(title="p95 latency per model", ylabel="ms", xlabel="")
(100 * err_daily).plot(ax=axes[1], marker="o", legend=False); axes[1].set(title="error rate per model", ylabel="%", xlabel="")
(daily[["input_tokens", "output_tokens"]] / 1e6).plot.bar(ax=axes[2], stacked=True, color=["#74c0fc", "#3b5bdb"])
axes[2].set(title="tokens per day", ylabel="million tokens", xlabel="")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

Three things jump out, which is what a dashboard is for: the gpt-4.1 error spike on one day, a steady latency
ordering between models, and input tokens stepping up from day 4 **with no change in traffic**.

## 5. Debugging "token usage doubled overnight" (`po32`)

Split input from output (different causes), find the change point, then group by every dimension you logged. The
one that explains the jump is the answer; here it is the prompt version.

In [ ]:
by_version = logs.groupby(["day", "prompt_version"])["input_tokens"].mean().unstack().round(0)
print(by_version.to_string())
share_v8 = logs.assign(v8=logs["prompt_version"] == "support-v8").groupby("day")["v8"].mean().round(2)
print("\nshare of traffic on support-v8:", share_v8.to_dict())
mean_in = logs.groupby("day")["input_tokens"].mean()
print(f"\nmean input tokens: {mean_in.iloc[:3].mean():.0f} before day 4, {mean_in.iloc[3:].mean():.0f} after")
assert mean_in.iloc[3:].mean() > 1.5 * mean_in.iloc[:3].mean()

"No deploy" rarely survives the evidence: a prompt rolled out by config, a retrieval `k` change, a re-index with
bigger chunks, a broken prompt-cache prefix, or an agent looping. Look at p99, not only the mean: a few runaway
requests can carry the whole increase.

## 6. Alerts worth having

Alert on **rates, percentiles and money** over a window, never on single events. Each alert needs an owner and a
linked runbook, or it gets muted within a month (`po24`, `pd07`).

| Alert | Condition (example) | Why |
|---|---|---|
| Latency | p95 > SLO for 5 min, per model and route | the user-facing promise |
| Errors by class | 429 rate > 2% for 5 min; 5xx rate > 1% | 429 = capacity, 5xx = provider health; different fixes |
| Cost per request | p99 cost up 30% on its baseline (or > 5x the median) | catches runaway loops and prompt growth that total spend hides |
| Daily spend | projected spend > budget | the finance conversation, early |
| Prompt-cache hit rate | cached-token ratio drops by half | someone changed the prompt prefix |
| Empty retrieval | rate > baseline + 3 sigma | ingestion or index failure; answers go ungrounded |
| Refusal / escalation rate | up 50% week over week | earliest quality signal you have |
| Fallback share | > 5% of traffic served degraded | the primary is sick even if errors look fine |

In [ ]:
def evaluate_alerts(df: pd.DataFrame, slo_p95_ms=5_000, max_err=0.05, cost_growth=1.3):
    fired = []
    for (d, m), g in df.groupby(["day", "model"]):
        p95, err = g["latency_ms"].quantile(0.95), (g["status"] != "ok").mean()
        if p95 > slo_p95_ms:
            fired.append((d, m, f"p95 {p95:,.0f} ms > {slo_p95_ms:,} ms"))
        if err > max_err:
            fired.append((d, m, f"error rate {err:.0%} > {max_err:.0%}"))
    days = sorted(df["day"].unique())
    base = df.loc[df["day"] == days[0], "cost_usd"].quantile(0.99)       # baseline: the first day's p99
    for d, g in df.groupby("day"):
        p99 = g["cost_usd"].quantile(0.99)
        if p99 > cost_growth * base:
            fired.append((d, "all", f"p99 cost/request ${p99:.5f} is {p99 / base:.1f}x the baseline"))
    return fired

alerts = evaluate_alerts(logs)
for a in alerts:
    print("ALERT", *a)
assert any("error rate" in msg and m == "gpt-4.1" for _, m, msg in alerts)
assert {d for d, m, msg in alerts if "cost" in msg} == {"09-04", "09-05", "09-06", "09-07"}

## 7. OpenTelemetry: the standard way to do section 3

OpenTelemetry (OTel) is the vendor-neutral standard for traces, metrics and logs. The concepts map one-to-one onto
the toy tracer: a **trace** of **spans** with parent ids, **attributes** (key-value pairs) and **events** on spans,
a **context** that follows the request (in-process via context variables, across services via the W3C
`traceparent` HTTP header), and an **exporter** that ships finished spans to a backend (Jaeger, Tempo, Datadog,
Langfuse, Phoenix...).

For LLM calls, the **GenAI semantic conventions** standardise attribute names, for example `gen_ai.operation.name`,
`gen_ai.request.model`, `gen_ai.response.model`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens`,
`gen_ai.response.finish_reasons` and the provider (`gen_ai.provider.name`, formerly `gen_ai.system`). They are still
marked as in development and have been renamed between versions, so pin a version and check the spec.

The `opentelemetry-api` package happens to be installed here (a dependency of `mcp`) but not the SDK. That shows a
design point: libraries instrument against the API, which is a **no-op** until the application installs the SDK and
an exporter.

In [ ]:
try:
    from opentelemetry import trace
    tracer = trace.get_tracer("chat-service")
    with tracer.start_as_current_span("llm.chat") as sp:
        sp.set_attribute("gen_ai.request.model", MODEL)
        print(f"span type: {type(sp).__name__}, recording: {sp.is_recording()}  <- no SDK installed, so a no-op")
except ImportError:
    print("opentelemetry-api is not installed; see the snippet below.")

With the SDK (`pip install opentelemetry-sdk opentelemetry-exporter-otlp`), wiring it up looks like this; auto-
instrumentation packages exist for the OpenAI SDK, LangChain, FastAPI and httpx (for example OpenLLMetry and
OpenInference):

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(endpoint="http://localhost:4318/v1/traces")))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("chat-service")

with tracer.start_as_current_span("chat gpt-4.1-mini") as span:
    span.set_attribute("gen_ai.operation.name", "chat")
    span.set_attribute("gen_ai.request.model", "gpt-4.1-mini")
    r = client.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    span.set_attribute("gen_ai.usage.input_tokens", r.usage.prompt_tokens)
    span.set_attribute("gen_ai.usage.output_tokens", r.usage.completion_tokens)
```

## 8. The tool landscape

| Tool | Shape | Good at | Consider |
|---|---|---|---|
| **LangSmith** | hosted (self-host on enterprise plans) | LangChain/LangGraph-native tracing, datasets, evals | framework gravity; data leaves your network |
| **Langfuse** | open source, self-hostable or cloud | traces, prompt management, evals, cost; OTel-compatible | you run Postgres + ClickHouse if self-hosted |
| **Arize Phoenix** | open source, runs locally; Arize cloud | OTel/OpenInference-based tracing and evals, great for notebooks | the hosted product is a separate tier |
| **Helicone** | proxy/gateway (change the base URL) or SDK | zero-code logging, caching, per-user cost | sits in the request path: must fail open |
| **W&B Weave, Braintrust** | hosted | evals and experiment tracking tied to traces | pricing and data residency |
| **Datadog, New Relic, Grafana** | general APM with LLM features | one pane with the rest of your services | LLM-specific depth varies |

The decision is mostly hosting and data residency, not features: traces contain your prompts and answers (`tl32`).

## 9. Drift and quality monitoring

**Drift** is "nothing changed on our side and it got worse": users ask new things, documents change, or a provider
updates a model behind an alias (`pd30`). Watch input distributions (length, topic, language, embedding clusters),
output distributions (length, refusal rate) and a sampled quality score. The **population stability index** (PSI)
is a simple, standard way to compare two distributions of one feature.

In [ ]:
def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-6
    a = np.histogram(actual, edges)[0] / len(actual) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

week1 = rng.lognormal(np.log(40), 0.5, 3_000)            # user question length in tokens
week2_same = rng.lognormal(np.log(40), 0.5, 3_000)
week2_drift = np.concatenate([rng.lognormal(np.log(40), 0.5, 2_100), rng.lognormal(np.log(160), 0.4, 900)])  # pasted documents
print(f"PSI week1 vs week2 (no change): {psi(week1, week2_same):.3f}")
print(f"PSI week1 vs week2 (drift)    : {psi(week1, week2_drift):.3f}   (rule of thumb: <0.1 stable, >0.25 investigate)")
assert psi(week1, week2_same) < 0.1 < 0.25 < psi(week1, week2_drift)

fig, ax = plt.subplots(figsize=(9, 3.4))
bins = np.logspace(np.log10(5), np.log10(800), 50)
ax.hist(week1, bins=bins, alpha=0.6, label="week 1", color="#74c0fc")
ax.hist(week2_drift, bins=bins, alpha=0.6, label="week 2", color="#e03131")
ax.set_xscale("log")
ax.set(xlabel="user question length (tokens, log scale)", ylabel="requests",
       title=f"Input drift: a new cluster of long pasted inputs (PSI {psi(week1, week2_drift):.2f})")
ax.legend(); plt.show()

For **quality**, run a continuous evaluation stream: deterministic checks on every response (schema valid,
citation exists, refusal detected), an LLM judge on a small **stratified** sample, and human review of the
disagreements; chart the scores per release and per segment (`pd32`). Pin model versions so a provider update is a
deliberate event, and re-run the frozen eval set nightly so slow drift shows up as a trend.

## Try it yourself

**1.** Which model and day explain most of the errors? Answer with one pandas expression that ranks (day, model)
pairs by error count.

In [ ]:
# Solution — try it yourself first, then run this
top = (logs[logs["status"] != "ok"].groupby(["day", "model"]).size().sort_values(ascending=False).head(3))
print(top.to_string())
assert top.index[0][1] == "gpt-4.1"

**2.** Extend the tracer: record an **event** (a timestamped message) on the current span, and make a failing tool
call mark its span as an error without hiding the exception.

In [ ]:
# Solution — try it yourself first, then run this
def add_event(name, **attrs):
    current_span.get().events.append((time.perf_counter(), name, attrs))

FINISHED.clear()
try:
    with Span("agent.run"):
        add_event("guardrail.checked", verdict="allow")
        with Span("tool.refund"):
            raise TimeoutError("payments API did not answer")
except TimeoutError:
    pass
for s in FINISHED:
    print(f"{s.name:<12} status={s.status:<6} attrs={s.attributes} events={[e[1] for e in s.events]}")
assert [s.status for s in FINISHED] == ["error", "error"]

**3.** Your finance team wants an alert when **cost per successful request** on any day rises 30% above the first
day's. Compute it (failed calls still cost money) and report which days would fire.

In [ ]:
# Solution — try it yourself first, then run this
per_day = logs.groupby("day").apply(lambda g: g["cost_usd"].sum() / (g["status"] == "ok").sum(), include_groups=False)
baseline = per_day.iloc[0]
print((per_day / baseline).round(2).to_string())
fire = list(per_day[per_day > 1.3 * baseline].index)
print("would fire on:", fire, "<- the days after the longer prompt shipped")
assert fire == ["09-04", "09-05", "09-06", "09-07"]

if OFFLINE:                      # reset the stand-in's knobs
    OFFLINE.latency = 0.0

## Say it in an interview

- **30 seconds:** "Every request gets a trace id that also appears in user-facing errors. Each LLM call is a span
  with model and version, prompt version, tokens (including cached), latency and TTFT, cost, status and error class,
  route and cache hit; retrieval spans carry chunk ids and scores; tool spans carry arguments and results; PII is
  redacted before anything is logged. From that: dashboards of p50/p95/p99 per stage, error rate by class, cost per
  request p50/p99, cache hit rate, empty-retrieval and refusal rates, plus a sampled quality score."
- **Standards and tools:** OpenTelemetry with the GenAI semantic conventions; Langfuse or Phoenix if data must stay
  in-house, LangSmith if you live in LangChain; the choice is mostly about hosting.
- **Alerts:** rates and percentiles over windows, error classes separated, p99 cost per request, cache hit rate,
  refusal/escalation rate; every alert has an owner and a runbook.
- **Debugging stories:** "tokens doubled" = split input/output, find the change point, group by prompt version,
  model, feature, tenant; "feels slow" = TTFT, not total latency.
- **Traps:** full prompts with PII in a third-party tracer; averaging percentiles; alerting on single errors;
  no retrieved-chunk ids in the logs; sampling that drops the failed traces.

## Next

- [05 · Latency percentiles and cost](05_latency_percentiles_p50_p95_and_cost.ipynb): what the dashboard numbers mean.
- [04 · FastAPI serving](04_fastapi_serving_and_streaming.ipynb): where the request id and middleware come from.
- [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb) · [agent evaluation](../12_evaluation/04_agent_evaluation.ipynb) · [LLM as judge](../12_evaluation/02_llm_as_judge_and_its_biases.ipynb).
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) · [AI tooling course](../../ai_tooling/index.html) · [interview bank](../../ai_interview_prep/index.html).